In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import kagglehub
import os

In [2]:

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [3]:
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
submission = pd.read_csv("/kaggle/input/competitions/titanic/gender_submission.csv")

In [4]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [7]:
train.isna().any(axis=1).sum()

np.int64(708)

Out of 891 enteries there are 708 rows that has nan data. So, we can't just delete rows with nan. This might be cause by one heavily missing columns. 

In [8]:
train.isna().sum().sort_values(ascending=False)

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64

Cabin has 687 nan values. So, Lets replace the nan with a default value to keep what little information it has.

In [9]:
train['Cabin'] = train['Cabin'].fillna('N')

The cabin number is "C123" Where "C" is the deck of the ship and "123" is the cabin number on that deck. 

And to avoid the too amny category problem and make the cabin more ordinal feature. We will split the feature into 

- Deck
- Cabin_num

Deck Order : `['A', 'B', 'C', 'D', 'E', 'F', 'G', 'N']`

In [10]:
train['Deck'] = train['Cabin'].astype(str).str[0]
train['Cabin_num'] = train['Cabin'].astype(str).str[1:]
train['Cabin_num'] = pd.to_numeric(train['Cabin_num'], errors='coerce')
train['Cabin_num'] = train['Cabin_num'].fillna(0).astype(int)
train[['Deck','Cabin_num']]

,Deck,Cabin_num
0,N,0
1,C,85
2,N,0
3,C,123
4,N,0
...,...,...
886,N,0
887,B,42
888,N,0
889,C,148


In [11]:
# train['Age'] = train['Age'].fillna(train['Age'].mean())
# train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

In [12]:
train.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin            0
Embarked         2
Deck             0
Cabin_num        0
dtype: int64

Do, the same for test data.

In [13]:
test['Cabin'] = test['Cabin'].fillna("N")
test['Deck'] = test['Cabin'].astype(str).str[0]
test['Cabin_num'] = test['Cabin'].astype(str).str[1:]
test['Cabin_num'] = pd.to_numeric(test['Cabin_num'], errors='coerce')
test['Cabin_num'] = test['Cabin_num'].fillna(0).astype(int)

test['Age'] = test['Age'].fillna(train['Age'].mean())
test['Embarked'] = test['Embarked'].fillna(train['Embarked'].mode()[0])
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Deck,Cabin_num
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,N,Q,N,0
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,N,S,N,0
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,N,Q,N,0
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,N,S,N,0
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,N,S,N,0


In [14]:
train.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin            0
Embarked         2
Deck             0
Cabin_num        0
dtype: int64

All the nan values are removed. Now, let check for duplicate rows.

In [15]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        891 non-null    object 
 11  Embarked     889 non-null    object 
 12  Deck         891 non-null    object 
 13  Cabin_num    891 non-null    int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 97.6+ KB


Lets create a new feature by combining SibSp, ParCh 

FamilySize = SibSp + Parch + 1

+1 reperesents themself. 

In [16]:
train['FamilySize'] = train['Parch'] + train['SibSp'] + 1 
test['FamilySize'] = test['Parch'] + test['SibSp'] + 1 

train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Deck,Cabin_num,FamilySize
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,N,S,N,0,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,C,85,2
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,N,S,N,0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,C,123,2
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,N,S,N,0,1


In [17]:
train.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'Deck', 'Cabin_num',
       'FamilySize'],
      dtype='object')

In [18]:
cat_feats = ['Sex','Embarked']
ord_feats = ["Pclass",'Deck']
num_feats = ['Age','SibSp','Parch','Fare','FamilySize',"Cabin_num"]

pclass_ord = [1,2,3]
deck_ord = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'T', 'N']

categories_list = [pclass_ord,deck_ord]

In [19]:
train[ord_feats] = train[ord_feats].astype('category')
train[cat_feats] = train[cat_feats].astype('category')

In [20]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   PassengerId  891 non-null    int64   
 1   Survived     891 non-null    int64   
 2   Pclass       891 non-null    category
 3   Name         891 non-null    object  
 4   Sex          891 non-null    category
 5   Age          714 non-null    float64 
 6   SibSp        891 non-null    int64   
 7   Parch        891 non-null    int64   
 8   Ticket       891 non-null    object  
 9   Fare         891 non-null    float64 
 10  Cabin        891 non-null    object  
 11  Embarked     889 non-null    category
 12  Deck         891 non-null    category
 13  Cabin_num    891 non-null    int64   
 14  FamilySize   891 non-null    int64   
dtypes: category(4), float64(2), int64(6), object(3)
memory usage: 80.9+ KB


In [21]:
train['Deck'].value_counts()

Deck
N    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [22]:
X = train[num_feats+cat_feats+ord_feats]
y = train['Survived']

In [23]:
import sklearn
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression 
from sklearn.model_selection import cross_val_score, cross_val_predict

num_transformer = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('scaler',StandardScaler())
    ])
cat_transformer = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ("cat_encoder",OneHotEncoder(handle_unknown='ignore'))
    ])
ord_transformer = Pipeline(
    steps=[
        ("ord_encoder",OrdinalEncoder(categories=categories_list))
    ])


preprocessor = ColumnTransformer(
    transformers=[
        ('num',num_transformer,num_feats),
        ("cat",cat_transformer,cat_feats),
        ('ord',ord_transformer,ord_feats)
    ]) 

clf = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',LogisticRegression(max_iter=1000))
])

cv = cross_val_score(clf,X,y,cv=5,scoring='accuracy')

print(cv.mean())

0.7833783190006904


In [24]:
clf.fit(X,y)
pred = clf.predict(test[num_feats+cat_feats+ord_feats])
sub = pd.DataFrame({
    'PassengerId':test['PassengerId'],
    'Survived': pred
})

sub.to_csv('submission.csv',index=False)
print("Submission successful !")

Submission successful !


### Back pocket: 
1. ~Change the train data filling nan with mean directly and do it in pipeline so that each cv fold train data has its own mean and not the train data's mean.~
2. Lets find the pattern in the ticket by grouping and finding family with same ticket number which also means they have the same fair price(which is the sum). 